# CoT Distillation v3 — Kaggle Pipeline

**v3 improvements over v2:**
- Answer-weighted loss during training (`answer_weight` ≥ 1.0 up-weights `#### N` tokens)
- `correct_and_propagate` in accuracy eval (fixes cascading arithmetic errors, not just single equations)
- Pythia-410M informativeness scorer (replaces GPT-2 in Stage 5b)
- Strict-parser accuracy column in eval output

Cells are idempotent — safe to re-run. Skip training sections if checkpoints already exist.

## 0 · Setup

In [1]:
# 0.1 — Clone OR pull the repo. Hard-resets to origin/BRANCH so stale local
# edits in the Kaggle ephemeral workspace can't block the pull.
import os, subprocess, pathlib

REPO_URL  = "https://github.com/rihembenabdallah18/COT_lab.git"
BRANCH    = "dev"     # switch to 'main' once v3 is merged
REPO_NAME = "COT_lab"
WORKDIR   = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle").exists() else pathlib.Path.cwd()
REPO_DIR  = WORKDIR / REPO_NAME

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--prune"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "clean", "-fd"], check=True)

os.chdir(REPO_DIR)
print("repo:", REPO_DIR)
subprocess.run(["git", "log", "--oneline", "-5"], check=True)

Cloning into '/kaggle/working/COT_lab'...


repo: /kaggle/working/COT_lab
6782483 fix path
136bd3e update fineTuning
ca636b7 update accuracy.py
f1c4502 Add online calculator decoding (Stage 4b) + accuracy cleanup
89e563f Match Ho et al. first-number evaluation protocol for baseline


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

In [2]:
# 0.2 — Install dependencies.
!pip install -q -r requirements.txt
!pip uninstall -y -q peft || true
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 4.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 564.8 kB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.1/314.1 kB 3.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 87.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 79.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 62.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 31.2 

In [6]:
# 0.3 — GPU check. Should print True + Tesla T4.
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
Tesla T4
VRAM: 15.6 GB


## 1 · Config — review and (optionally) override

In [7]:
import yaml, pathlib, json

CFG_PATH = pathlib.Path("config/config.yaml")
cfg = yaml.safe_load(CFG_PATH.read_text())

# ── v3 overrides ────────────────────────────────────────────────────────────
OVERRIDES = {
    # Answer-weighted loss: multiply #### suffix token loss by this factor.
    # 1.0 = standard behaviour (same as v2). 3.0 is a good starting point.
    # Only takes effect if src/train/finetune.py supports answer_weight.
    "answer_weight": 3.0,

    # Pythia-410M informativeness scorer in Stage 5b.
    # Set to "gpt2" to reproduce v2 results.
    "info_scorer_model": "EleutherAI/pythia-410m",

    # Propagate arithmetic corrections across dependent equations in Stage 5a.
    "calculator_propagate": True,
}
# ────────────────────────────────────────────────────────────────────────────

for k, v in OVERRIDES.items():
    cfg[k] = v

if OVERRIDES:
    CFG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print("applied v3 overrides:", list(OVERRIDES.keys()))

DISPLAY_KEYS = [
    "model_name", "learning_rate", "weight_decay", "warmup_ratio",
    "num_epochs", "early_stopping_patience", "max_input_length",
    "max_target_length", "batch_size", "gradient_accumulation_steps",
    "fp16", "inference_num_beams", "inference_no_repeat_ngram_size",
    "inference_repetition_penalty", "inference_max_new_tokens", "seed",
    "answer_weight", "info_scorer_model", "calculator_propagate",
]
print(json.dumps({k: cfg.get(k) for k in DISPLAY_KEYS}, indent=2))

applied v3 overrides: ['answer_weight', 'info_scorer_model', 'calculator_propagate']
{
  "model_name": "google/flan-t5-base",
  "learning_rate": 5e-05,
  "weight_decay": 0.05,
  "warmup_ratio": 0.1,
  "num_epochs": 12,
  "early_stopping_patience": 2,
  "max_input_length": 512,
  "max_target_length": 512,
  "batch_size": 4,
  "gradient_accumulation_steps": 8,
  "fp16": true,
  "inference_num_beams": 4,
  "inference_no_repeat_ngram_size": 4,
  "inference_repetition_penalty": 1.15,
  "inference_max_new_tokens": 512,
  "seed": 42,
  "answer_weight": 3.0,
  "info_scorer_model": "EleutherAI/pythia-410m",
  "calculator_propagate": true
}


## 2 · Project status (run any time)

In [8]:
!python -m src.status

  COT_lab project status
Stage         Run                  Status     Duration  Headline                                                                                              
------------  -------------------  ---------  --------  ------------------------------------------------------------------------------------------------------
02            filter               completed  00:00:00  A=7473 B=3389 C=2635 direct=7473                                                                      
03_train      student_direct_ft    completed  00:39:36  best_eval_loss=0.867 @ epoch 7.990487514863258                                                        
03_train      student_set_a        completed  00:57:09  best_eval_loss=0.922 @ epoch 7.990487514863258                                                        
03_train      student_set_b        completed  00:25:13  best_eval_loss=0.922 @ epoch 6.994764397905759                                                        
03_train      student

## 3 · Stage 1 — Download GSM8K + Ho et al. teacher CoTs

In [9]:
!bash scripts/01_download.sh

[gsm8k] downloading via datasets.load_dataset('gsm8k', 'main')
Generating test split: 100%|█████| 1319/1319 [00:00<00:00, 269604.63 examples/s]
[gsm8k] saved 7473 train, 1319 test -> /kaggle/working/COT_lab/data/raw/gsm8k
[ho] downloading shared folder zip (~924 MB) to /tmp/tmpx_qarz56/release.zip
[curl] curl -fSL --retry 3 -o /tmp/tmpx_qarz56/release.zip https://www.dropbox.com/sh/hwcncpyomx87h20/AACqgVdd-ZzBQ3ncJcKqw0cVa?dl=1
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   140  100   140    0     0    788      0 --:--:-- --:--:-- --:--:--   786
100    17  100    17    0     0     24      0 --:--:-- --:--:-- --:--:--    24
100  880M  100  880M    0     0  64.6M      0  0:00:13  0:00:13 --:--:-- 72.9M
[ho] extracting teacher_completion_data.tar.gz from zip
[ho] opening tarball /tmp/tmpx_qarz56/teacher_completion_data.tar.gz
[ho] extracting teacher_completion_data/B_text-d

## 4 · Stage 2 — Build training sets (A / B / C)

In [10]:
!bash scripts/02_filter.sh

GSM8K train rows: 7473
  Direct FT      : 7473 -> /kaggle/working/COT_lab/data/processed/direct_ft.jsonl
  Set A (no flt) : 7473 -> /kaggle/working/COT_lab/data/processed/set_a_nofilter.jsonl
  Set B (Magist) : 3389 -> /kaggle/working/COT_lab/data/processed/set_b_magister.jsonl
  Set C (calc.)  : 2635 -> /kaggle/working/COT_lab/data/processed/set_c_calculator.jsonl
  skipped: no_teacher=0 unparseable_gold=0 unparseable_teacher_pred=78
  Set B keep rate (of A): 45.3%
  Set C keep rate (of A): 35.3%
  CoTs the calculator edited: 676 / 7473

--- Set B / Set C contingency ---
  in both    : 2585
  Set C only : 50   (chains rescued by calculator)
  Set B only : 804   (calc broke a previously-correct answer)
  in neither : 4034

--- 2 example records, Set A ---
{
  "sample_index": 0,
  "question": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natali...",
  "cot": "In April, Natalia sold 48 clips. \nIn May, she sold half as 

In [11]:
# Verify set sizes — expected: A≈7473, B≈3389, C≈2635, direct_ft≈7473
import pathlib, json

processed = pathlib.Path("data/processed")
for split, filename in [
    ("train_set_a",    "set_a_nofilter.jsonl"),
    ("train_set_b",    "set_b_magister.jsonl"),
    ("train_set_c",    "set_c_calculator.jsonl"),
    ("train_direct_ft","direct_ft.jsonl"),
]:
    p = processed / filename
    if p.exists():
        n = sum(1 for l in p.open() if l.strip())
        print(f"{split:20s}: {n:6d} examples")
    else:
        print(f"{split:20s}: MISSING")

train_set_a         :   7473 examples
train_set_b         :   3389 examples
train_set_c         :   2635 examples
train_direct_ft     :   7473 examples


In [12]:
# Unit tests
!python -m pytest tests/ -q

...........................                                              [100%]
27 passed in 0.05s


## 5 · Stage 3 — Fine-tune students

> **v3 change**: if `answer_weight > 1.0` is supported by `src/train/finetune.py`,
> the `#### N` suffix tokens receive `answer_weight`× the standard cross-entropy loss.
> This re-anchors the gradient signal at the correct answer and narrows the CoT vs Direct FT accuracy gap.

In [13]:
# Storage cleanup helper — strips optimizer/scheduler/rng state post-training
import json, pathlib, shutil

CKPT_ROOT = pathlib.Path("outputs/checkpoints")
_TRAIN_STATE = ("optimizer.pt", "scheduler.pt")

def _dir_size(p):
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def _fmt(b):
    return f"{b/1e9:.2f} GB" if b >= 1e9 else f"{b/1e6:.0f} MB"

def cleanup_run(run_name, keep_best_only=True):
    run_dir = CKPT_ROOT / run_name
    if not run_dir.exists():
        print(f"[cleanup:{run_name}] not found, skipping."); return
    ckpts = sorted(run_dir.glob("checkpoint-*"),
                   key=lambda p: int(p.name.split("-")[-1]))
    if not ckpts:
        print(f"[cleanup:{run_name}] no checkpoints found."); return
    before = _dir_size(run_dir)
    best_name = None
    state_path = ckpts[-1] / "trainer_state.json"
    if state_path.exists():
        bmc = json.loads(state_path.read_text()).get("best_model_checkpoint")
        if bmc:
            best_name = pathlib.Path(bmc).name
    kept = []
    for ckpt in ckpts:
        if keep_best_only and best_name and ckpt.name != best_name:
            shutil.rmtree(ckpt)
            print(f"[cleanup:{run_name}] removed non-best {ckpt.name}"); continue
        for fname in _TRAIN_STATE:
            f = ckpt / fname
            if f.exists(): f.unlink()
        for f in ckpt.glob("rng_state*.pth"):
            f.unlink()
        kept.append(ckpt.name)
    after = _dir_size(run_dir)
    print(f"[cleanup:{run_name}] kept {kept} | freed {_fmt(before - after)} "
          f"({_fmt(before)} → {_fmt(after)})")

print("cleanup_run() ready.")

cleanup_run() ready.


### 5.1 — Smoke test (200 ex, 1 epoch, ~10 min on T4)

In [14]:
!bash scripts/03_train_smoke.sh

2026-05-15 20:50:21.589918: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778878221.813964     215 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778878221.875759     215 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778878222.417869     215 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778878222.417955     215 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778878222.417964     215 computation_placer.cc:177] computation placer alr

### 5.2 — Direct FT (Q→A only, no CoT)

In [15]:
!bash scripts/03_train_direct_ft.sh

2026-05-15 20:52:07.822662: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778878327.847337     817 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778878327.855150     817 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778878327.874605     817 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778878327.874640     817 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778878327.874644     817 computation_placer.cc:177] computation placer alr

In [ ]:
cleanup_run("student_direct_ft")

### 5.3 — Set B (Magister filter)

In [ ]:
!bash scripts/03_train_set_b.sh

In [ ]:
cleanup_run("student_set_b")

### 5.4 — Set C (calculator-corrected, process-aware)

In [ ]:
!bash scripts/03_train_set_c.sh

In [ ]:
cleanup_run("student_set_c")

### 5.5 — Set A (no filter, longest run)

In [ ]:
!bash scripts/03_train_set_a.sh

In [ ]:
cleanup_run("student_set_a")

## 6 · Stage 4 — Inference on the GSM8K test set

In [ ]:
!bash scripts/04_inference.sh
!python -m src.status

In [ ]:
# Quick sanity check: confirm all 6 generation files exist and are non-empty
import pathlib, json

GEN_DIR = pathlib.Path("outputs/generations")
CONDITIONS = ["baseline", "baseline_greedy", "student_direct_ft", "student_set_a", "student_set_b", "student_set_c"]
for c in CONDITIONS:
    p = GEN_DIR / f"{c}.jsonl"
    if p.exists():
        n = sum(1 for l in p.open() if l.strip())
        sample = json.loads(next(l for l in p.open() if l.strip()))
        fmt = "has ####" if "####" in sample.get("generated_cot", "") else "no ####"
        print(f"{c:25s}: {n:5d} examples | {fmt}")
    else:
        print(f"{c:25s}: MISSING")

## 6b · Stage 4b — Online Calculator Decoding

Runs the same checkpoints as Stage 4 but with **token-by-token greedy generation** that intercepts each completed equation and injects the correct result before the next token is sampled. This prevents wrong values from propagating through downstream reasoning steps.

- Output: `outputs/generations/student_set_{a,b,c}_oc.jsonl`
- Runtime: ~40 min per condition (~2 hours total)
- Only meaningful for CoT student conditions — baseline and direct_ft don't emit equations

In [ ]:
!bash scripts/04b_inference_online_calc.sh

In [ ]:
# Sanity check: confirm _oc files exist
import pathlib, json

GEN_DIR = pathlib.Path('outputs/generations')
for cond in ['student_set_a_oc', 'student_set_b_oc', 'student_set_c_oc']:
    p = GEN_DIR / f'{cond}.jsonl'
    if p.exists():
        n = sum(1 for l in p.open() if l.strip())
        sample = json.loads(next(l for l in p.open() if l.strip()))
        has_eq = any(op in sample.get('generated_cot', '') for op in [' + ', ' - ', ' * ', ' / '])
        print(f'{cond:30s}: {n:5d} examples | {"has equations" if has_eq else "no equations"}')
    else:
        print(f'{cond:30s}: MISSING')

## 7 · Stage 5 — Evaluation

### 7.1 — Stage 5a: Accuracy

> **v3 change**: `calculator_propagate=True` means the accuracy script uses
> `correct_and_propagate` instead of `correct_equations`, fixing cascading
> arithmetic errors across dependent steps.

In [ ]:
!bash scripts/05a_accuracy.sh

In [ ]:
# Show accuracy table (accuracy + accuracy_w_calc for student_set_* conditions)
import csv, pathlib

acc_path = pathlib.Path("outputs/eval_results/accuracy.csv")
if acc_path.exists():
    with acc_path.open() as f:
        rows = list(csv.DictReader(f))
    header = list(rows[0].keys())
    col_w  = max(len(h) for h in header) + 2
    print("  ".join(h.ljust(col_w) for h in header))
    print("-" * (col_w * len(header)))
    for r in rows:
        print("  ".join(str(r[h]).ljust(col_w) for h in header))
else:
    print("accuracy.csv not found — run 05a_accuracy.sh first.")

### 7.2 — Stage 5b: ReCEval

> **v3 change**: informativeness scorer uses `EleutherAI/pythia-410m` instead of GPT-2.
> Pass `--info-scorer gpt2` to reproduce v2 scores for comparison.

In [ ]:
!bash scripts/05b_receval.sh

In [ ]:
# Show ReCEval summary
import csv, pathlib

re_path = pathlib.Path("outputs/eval_results/receval_summary.csv")
if re_path.exists():
    with re_path.open() as f:
        rows = list(csv.DictReader(f))
    COLS = ["condition", "n", "intra_mean", "intra_std", "inter_mean", "inter_std",
            "info_mean", "info_std"]
    cols = [c for c in COLS if c in rows[0]]
    print("  ".join(c[:12].ljust(13) for c in cols))
    print("-" * (14 * len(cols)))
    for r in rows:
        def _fmt(v):
            try: return f"{float(v):.4f}"
            except: return str(v)
        print("  ".join(_fmt(r.get(c, "")).ljust(13) for c in cols))
else:
    print("receval_summary.csv not found — run 05b_receval.sh first.")

In [ ]:
# Show current run-card status
!python -m src.status

## 8 · Persist outputs across sessions

In [ ]:
# 8.1 — Package outputs/ + data/processed/ and push to a Kaggle Dataset.
import json, os, pathlib, shutil, subprocess

ON_KAGGLE           = pathlib.Path("/kaggle").exists()
KAGGLE_USERNAME     = os.environ.get("KAGGLE_USERNAME", "")
BACKUP_DATASET_SLUG = "cot-lab-v3-outputs"

if not ON_KAGGLE:
    print("not on Kaggle — skipping dataset upload.")
elif not KAGGLE_USERNAME:
    print("KAGGLE_USERNAME not set — skipping. (On Kaggle this is normally injected.)")
else:
    STAGE_DIR = pathlib.Path("/kaggle/working/_dataset_payload")
    STAGE_DIR.mkdir(parents=True, exist_ok=True)
    for src in ["outputs/checkpoints", "outputs/generations", "outputs/runs",
                "outputs/eval_results", "data/processed"]:
        src_path = pathlib.Path(src)
        if src_path.exists():
            dst = STAGE_DIR / src_path.name
            if dst.exists(): shutil.rmtree(dst)
            shutil.copytree(src_path, dst)
            print("staged:", src, "->", dst)

    meta = {
        "title":    "COT_lab v3 outputs",
        "id":       f"{KAGGLE_USERNAME}/{BACKUP_DATASET_SLUG}",
        "licenses": [{"name": "CC0-1.0"}],
    }
    (STAGE_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

    res = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(STAGE_DIR),
         "-m", "v3 outputs", "-r", "zip"],
        capture_output=True, text=True,
    )
    if res.returncode != 0 and "Not Found" in (res.stderr + res.stdout):
        res = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(STAGE_DIR), "-r", "zip"],
            capture_output=True, text=True,
        )
    print(res.stdout)
    if res.returncode != 0:
        print("STDERR:", res.stderr)

In [ ]:
# 8.2 — Fallback: zip outputs for manual download
import pathlib, subprocess

ARCHIVE = pathlib.Path("/kaggle/working/cot_lab_v3_outputs.tar.gz") \
    if pathlib.Path("/kaggle").exists() else pathlib.Path("cot_lab_v3_outputs.tar.gz")

paths = [p for p in ["outputs/checkpoints", "outputs/generations",
                     "outputs/runs", "outputs/eval_results",
                     "data/processed"] if pathlib.Path(p).exists()]
if not paths:
    print("nothing to archive yet.")
else:
    subprocess.run(["tar", "-czf", str(ARCHIVE), *paths], check=True)
    size_mb = ARCHIVE.stat().st_size / 1e6
    print(f"wrote {ARCHIVE} ({size_mb:.1f} MB)")
    for pp in paths:
        print(" -", pp)